In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(1)

In [7]:
df = pd.DataFrame({
    "subject": ["S1", "S1", "S2", "S2"],
    "condition": ["A", "B", "A", "B"],
    "RT": [520, 610, 450, 700],
    "accuracy": [1, 0, 1, 1]
})

> apply() operates along an axis of a DataFrame.

- axis=0 is column-wise (default).
- axis=1 is row-wise.

It is more flexible but slower than vectorized methods.   
Note: transform() preserves index alignment and is typically clearer and faster.

In [17]:
# Column-wise:
df_colwise = df[["RT"]].apply(lambda col: col - col.mean(), axis=0)
print("Mean-centered RT per column:\n", df_colwise)

# Row-wise
df_rowise = df["efficiency"] = df.apply(
    lambda row: row["RT"] / row["accuracy"] if row["accuracy"] != 0 else np.nan,
    axis=1
)
print("\nEfficiency score (RT / accuracy):\n", df_rowise)

# Note: row-wise apply() is slow because it operates at Python level per row. Whenever possible, replace with vectorized expressions.
df_vectorized = df["efficiency"] = df["RT"] / df["accuracy"].replace(0, np.nan)
print("\nEfficiency score (vectorized):\n", df_vectorized)

Mean-centered RT per column:
       RT
0  -50.0
1   40.0
2 -120.0
3  130.0

Efficiency score (RT / accuracy):
 0    520.0
1      NaN
2    450.0
3    700.0
dtype: float64

Efficiency score (vectorized):
 0    520.0
1      NaN
2    450.0
3    700.0
dtype: float64


> Series.map() applies a function or mapping to each element of a single Series.

It is appropriate for recoding, labeling, or simple element-wise transformations.

In [31]:
# Categorizing RTs:
def categorize_rt(rt):
    if rt < 500:
        return "fast"
    elif rt < 650:
        return "medium"
    else:
        return "slow"

df["RT_category"] = df["RT"].map(categorize_rt)
print("Categorized df:\n", df.head())

# Vectorized (alternative):
df["RT_category"] = pd.cut(
    df["RT"],
    bins=[-np.inf, 500, 650, np.inf],
    labels=["fast", "medium", "slow"]
)

# Dictionary mapping (alternative):
condition_map = {"A": "control", "B": "experimental"}

df["condition_label"] = df["condition"].map(condition_map)
print("Conditoin-labeled df:\n", df.head())

Categorized df:
   subject condition   RT  accuracy  efficiency RT_category condition_label  \
0      S1         A  520         1       520.0      medium         control   
1      S1         B  610         0         NaN      medium    experimental   
2      S2         A  450         1       450.0        fast         control   
3      S2         B  700         1       700.0        slow    experimental   

       RT_z  
0 -0.707107  
1  0.707107  
2 -0.707107  
3  0.707107  
Conditoin-labeled df:
   subject condition   RT  accuracy  efficiency RT_category condition_label  \
0      S1         A  520         1       520.0      medium         control   
1      S1         B  610         0         NaN      medium    experimental   
2      S2         A  450         1       450.0        fast         control   
3      S2         B  700         1       700.0        slow    experimental   

       RT_z  
0 -0.707107  
1  0.707107  
2 -0.707107  
3  0.707107  


Note: for high-performance thresholding, np.select() or pd.cut() is usually preferable.

In [30]:
# Let's compute z-score per subject using transform.
# Z-score definition: z = (x − μ_subject) / σ_subject

# Using groupby combined with apply:
df["RT_z"] = df.groupby("subject")["RT"].transform(
    lambda x: (x - x.mean()) / x.std(ddof=1) # delta degrees of freedom = 1
)
print("Z-scores added:\n", df.head())

Z-scores added:
   subject condition   RT  accuracy  efficiency RT_category condition_label  \
0      S1         A  520         1       520.0      medium         control   
1      S1         B  610         0         NaN      medium    experimental   
2      S2         A  450         1       450.0        fast         control   
3      S2         B  700         1       700.0        slow    experimental   

       RT_z  
0 -0.707107  
1  0.707107  
2 -0.707107  
3  0.707107  


Vectorized operations (Pandas/NumPy):
   - Reduce interpreter overhead.
   - Exploit low-level optimizations (SIMD, cache locality).
   - Produce more concise and declarative code.
   - Reduce indexing errors and improve reproducibility.

Basicly they are often more effective than loops.